# 06b: CNN Development - Kaggle Cloud Training

Simplified version for P100 GPU training on Kaggle.

**Usage:** `./scripts/kaggle_full_pipeline.sh`

In [ ]:
# Import libraries
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score
)

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow: {tf.__version__}")
print(f"GPUs: {len(tf.config.list_physical_devices('GPU'))}")

In [ ]:
# Configure Kaggle paths
DATA_DIR = Path('/kaggle/input/data')
PROCESSED_DIR = Path('/kaggle/input/nih-chest-xray-splits')
OUTPUTS_DIR = Path('/kaggle/working/outputs')
MODELS_DIR = Path('/kaggle/working/models')
FIGURES_DIR = OUTPUTS_DIR / 'figures'

# Create output directories
OUTPUTS_DIR.mkdir(exist_ok=True, parents=True)
MODELS_DIR.mkdir(exist_ok=True, parents=True)
FIGURES_DIR.mkdir(exist_ok=True, parents=True)
(OUTPUTS_DIR / 'reports').mkdir(exist_ok=True, parents=True)

print(f"Data: {DATA_DIR}")
print(f"Splits: {PROCESSED_DIR}")
print(f"Models: {MODELS_DIR}")

In [ ]:
# CNN Configuration
CONFIG = {
    'img_height': 224,
    'img_width': 224,
    'channels': 3,
    'batch_size': 32,
    'epochs': 50,
    'learning_rate': 0.001,
    'filters': [32, 64, 128, 256],
    'dense_units': 512,
    'dropout_rate': 0.5,
    'l2_reg': 0.0001,
    'early_stopping_patience': 10,
    'reduce_lr_patience': 5,
    'num_classes': 14,
    'use_sample': True,
    'sample_size': 5000,
    'random_state': 42
}

print(json.dumps(CONFIG, indent=2))

In [ ]:
# Load data splits
train_df = pd.read_csv(PROCESSED_DIR / 'train_split.csv')
val_df = pd.read_csv(PROCESSED_DIR / 'val_split.csv')
test_df = pd.read_csv(PROCESSED_DIR / 'test_split.csv')

print(f"Train: {len(train_df):,}")
print(f"Val:   {len(val_df):,}")
print(f"Test:  {len(test_df):,}")

# Load preprocessing config
with open(PROCESSED_DIR / 'preprocessing_config.json', 'r') as f:
    prep_config = json.load(f)

disease_classes = prep_config['disease_classes']
class_weights_dict = prep_config['class_weights']

print(f"\nDisease classes: {len(disease_classes)}")

In [ ]:
# Convert local paths to Kaggle paths
# Local:  data/raw/images_001/images/00000001_000.png
# Kaggle: /kaggle/input/data/images_001/00000001_000.png

def update_kaggle_image_paths(df):
    def get_kaggle_path(local_path):
        path_obj = Path(local_path)
        filename = path_obj.name
        parent = path_obj.parent  # images/
        chunk_dir = parent.parent  # images_001/
        chunk_name = chunk_dir.name
        return f'/kaggle/input/data/{chunk_name}/{filename}'
    
    df['full_path'] = df['full_path'].apply(get_kaggle_path)
    return df

train_df = update_kaggle_image_paths(train_df)
val_df = update_kaggle_image_paths(val_df)
test_df = update_kaggle_image_paths(test_df)

print(f"Example path: {train_df['full_path'].iloc[0]}")

In [ ]:
# Diagnostic: Verify paths exist on Kaggle
import os

print("=" * 60)
print("PATH DIAGNOSTIC")
print("=" * 60)

# Check what's in /kaggle/input/
input_dir = Path('/kaggle/input')
print(f"\nAvailable datasets in /kaggle/input/:")
for item in sorted(input_dir.iterdir()):
    print(f"  - {item.name}")

# Check NIH dataset structure
print(f"\nNIH dataset directory: {DATA_DIR}")
print(f"Exists: {DATA_DIR.exists()}")

if DATA_DIR.exists():
    print(f"\nContents of {DATA_DIR}:")
    for item in sorted(DATA_DIR.iterdir())[:20]:  # Show first 20 items
        item_type = 'DIR' if item.is_dir() else 'FILE'
        print(f"  [{item_type}] {item.name}")
    
    # Check for image subdirectories
    subdirs = [d for d in DATA_DIR.iterdir() if d.is_dir() and d.name.startswith('images_')]
    if subdirs:
        print(f"\nFound {len(subdirs)} image subdirectories")
        first_subdir = sorted(subdirs)[0]
        print(f"\nContents of {first_subdir.name}/ (first 10 items):")
        for item in sorted(first_subdir.iterdir())[:10]:
            item_type = 'DIR' if item.is_dir() else 'FILE'
            size = f"{item.stat().st_size:,} bytes" if item.is_file() else ""
            print(f"  [{item_type}] {item.name} {size}")

# Check first image path
first_path = train_df['full_path'].iloc[0]
print(f"\nFirst image path: {first_path}")
print(f"Exists: {os.path.exists(first_path)}")

print("=" * 60)

In [ ]:
# Sample dataset if configured
if CONFIG['use_sample']:
    sample_size = CONFIG['sample_size']
    train_df = train_df.sample(n=min(sample_size, len(train_df)), random_state=42)
    val_df = val_df.sample(n=min(sample_size // 5, len(val_df)), random_state=42)
    test_df = test_df.sample(n=min(sample_size // 5, len(test_df)), random_state=42)
    
    print(f"⚠️  Using sample mode:")
    print(f"Train: {len(train_df):,}")
    print(f"Val:   {len(val_df):,}")
    print(f"Test:  {len(test_df):,}")

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
# Create ImageDataGenerators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

print("ImageDataGenerators created")

In [ ]:
# Create flow_from_dataframe generators
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='full_path',
    y_col=disease_classes,
    target_size=(CONFIG['img_height'], CONFIG['img_width']),
    batch_size=CONFIG['batch_size'],
    class_mode='raw',
    shuffle=True,
    seed=CONFIG['random_state']
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='full_path',
    y_col=disease_classes,
    target_size=(CONFIG['img_height'], CONFIG['img_width']),
    batch_size=CONFIG['batch_size'],
    class_mode='raw',
    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='full_path',
    y_col=disease_classes,
    target_size=(CONFIG['img_height'], CONFIG['img_width']),
    batch_size=CONFIG['batch_size'],
    class_mode='raw',
    shuffle=False
)

print(f"\nTrain batches: {len(train_generator)}")
print(f"Val batches:   {len(val_generator)}")
print(f"Test batches:  {len(test_generator)}")

In [ ]:
# Build CNN architecture
def build_custom_cnn(input_shape, num_classes, config):
    l2_reg = keras.regularizers.l2(config['l2_reg'])
    
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        # Conv Block 1: 32 filters
        layers.Conv2D(config['filters'][0], (3, 3), activation='relu', padding='same', kernel_regularizer=l2_reg),
        layers.Conv2D(config['filters'][0], (3, 3), activation='relu', padding='same', kernel_regularizer=l2_reg),
        layers.MaxPooling2D((2, 2)),
        layers.BatchNormalization(),
        
        # Conv Block 2: 64 filters
        layers.Conv2D(config['filters'][1], (3, 3), activation='relu', padding='same', kernel_regularizer=l2_reg),
        layers.Conv2D(config['filters'][1], (3, 3), activation='relu', padding='same', kernel_regularizer=l2_reg),
        layers.MaxPooling2D((2, 2)),
        layers.BatchNormalization(),
        
        # Conv Block 3: 128 filters
        layers.Conv2D(config['filters'][2], (3, 3), activation='relu', padding='same', kernel_regularizer=l2_reg),
        layers.Conv2D(config['filters'][2], (3, 3), activation='relu', padding='same', kernel_regularizer=l2_reg),
        layers.MaxPooling2D((2, 2)),
        layers.BatchNormalization(),
        
        # Conv Block 4: 256 filters
        layers.Conv2D(config['filters'][3], (3, 3), activation='relu', padding='same', kernel_regularizer=l2_reg),
        layers.Conv2D(config['filters'][3], (3, 3), activation='relu', padding='same', kernel_regularizer=l2_reg),
        layers.MaxPooling2D((2, 2)),
        layers.BatchNormalization(),
        
        # Dense layers
        layers.Flatten(),
        layers.Dense(config['dense_units'], activation='relu', kernel_regularizer=l2_reg),
        layers.Dropout(config['dropout_rate']),
        
        # Output layer
        layers.Dense(num_classes, activation='sigmoid')
    ])
    
    return model

print("CNN architecture function defined")

In [ ]:
# Build and compile model
input_shape = (CONFIG['img_height'], CONFIG['img_width'], CONFIG['channels'])
model = build_custom_cnn(input_shape, CONFIG['num_classes'], CONFIG)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CONFIG['learning_rate']),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.AUC(name='auc'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall')
    ]
)

model.summary()
print(f"\nTotal parameters: {model.count_params():,}")

In [ ]:
# Configure callbacks
callback_list = [
    callbacks.ModelCheckpoint(
        filepath=str(MODELS_DIR / 'cnn_custom_best.keras'),
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=CONFIG['early_stopping_patience'],
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=CONFIG['reduce_lr_patience'],
        min_lr=1e-7,
        verbose=1
    ),
    callbacks.CSVLogger(
        filename=str(OUTPUTS_DIR / 'reports' / '06_cnn_training_history.csv'),
        append=False
    )
]

print("Callbacks configured")

In [ ]:
# Train model
history = model.fit(
    train_generator,
    epochs=CONFIG['epochs'],
    validation_data=val_generator,
    callbacks=callback_list,
    verbose=1
)

print("\nTraining complete")

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(history.history['loss'], label='Train')
axes[0, 0].plot(history.history['val_loss'], label='Val')
axes[0, 0].set_title('Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(history.history['auc'], label='Train')
axes[0, 1].plot(history.history['val_auc'], label='Val')
axes[0, 1].set_title('AUC')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(history.history['precision'], label='Train')
axes[1, 0].plot(history.history['val_precision'], label='Val')
axes[1, 0].set_title('Precision')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(history.history['recall'], label='Train')
axes[1, 1].plot(history.history['val_recall'], label='Val')
axes[1, 1].set_title('Recall')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '06_cnn_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("Training history plotted")

In [ ]:
# Evaluate on test set
best_model = keras.models.load_model(MODELS_DIR / 'cnn_custom_best.keras')

test_results = best_model.evaluate(test_generator, verbose=1)

print(f"\nTest Results:")
print(f"Loss:      {test_results[0]:.4f}")
print(f"Accuracy:  {test_results[1]:.4f}")
print(f"AUC:       {test_results[2]:.4f}")
print(f"Precision: {test_results[3]:.4f}")
print(f"Recall:    {test_results[4]:.4f}")

In [ ]:
# Per-disease metrics
y_pred_proba = best_model.predict(test_generator, verbose=1)
y_pred = (y_pred_proba > 0.5).astype(int)
y_true = test_generator.labels[:len(y_pred)]

print(f"\n{'Disease':<20} {'AUC':>8} {'F1':>8} {'Precision':>10} {'Recall':>8}")
print("-" * 60)

per_disease_results = {}

for i, disease in enumerate(disease_classes):
    y_true_disease = y_true[:, i]
    y_pred_disease = y_pred[:, i]
    y_proba_disease = y_pred_proba[:, i]
    
    f1 = f1_score(y_true_disease, y_pred_disease, zero_division=0)
    precision = precision_score(y_true_disease, y_pred_disease, zero_division=0)
    recall = recall_score(y_true_disease, y_pred_disease, zero_division=0)
    
    if len(np.unique(y_true_disease)) > 1:
        auc = roc_auc_score(y_true_disease, y_proba_disease)
    else:
        auc = np.nan
    
    per_disease_results[disease] = {
        'auc': float(auc) if not np.isnan(auc) else None,
        'f1': float(f1),
        'precision': float(precision),
        'recall': float(recall)
    }
    
    print(f"{disease:<20} {auc:>8.3f} {f1:>8.3f} {precision:>10.3f} {recall:>8.3f}")

print("-" * 60)
avg_auc = np.nanmean([r['auc'] for r in per_disease_results.values() if r['auc'] is not None])
avg_f1 = np.mean([r['f1'] for r in per_disease_results.values()])
avg_precision = np.mean([r['precision'] for r in per_disease_results.values()])
avg_recall = np.mean([r['recall'] for r in per_disease_results.values()])

print(f"{'AVERAGE':<20} {avg_auc:>8.3f} {avg_f1:>8.3f} {avg_precision:>10.3f} {avg_recall:>8.3f}")

In [ ]:
# Save results
results = {
    'config': CONFIG,
    'test_performance': {
        'overall': {
            'loss': float(test_results[0]),
            'accuracy': float(test_results[1]),
            'auc': float(test_results[2]),
            'precision': float(test_results[3]),
            'recall': float(test_results[4])
        },
        'per_disease': per_disease_results,
        'averages': {
            'auc': float(avg_auc),
            'f1': float(avg_f1),
            'precision': float(avg_precision),
            'recall': float(avg_recall)
        }
    }
}

with open(OUTPUTS_DIR / 'reports' / '06_cnn_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved")

In [ ]:
# Package results for download
import shutil

download_dir = Path('/kaggle/working/download')
download_dir.mkdir(exist_ok=True)

# Copy model
shutil.copy(MODELS_DIR / 'cnn_custom_best.keras', download_dir / 'cnn_custom_best.keras')

# Copy reports and figures
shutil.copytree(OUTPUTS_DIR / 'reports', download_dir / 'reports', dirs_exist_ok=True)
shutil.copytree(FIGURES_DIR, download_dir / 'figures', dirs_exist_ok=True)

total_size = sum(f.stat().st_size for f in download_dir.rglob('*') if f.is_file()) / (1024**2)
print(f"\nPackaged for download: {total_size:.1f} MB")
print(f"Location: /kaggle/working/download/")